# DantinoX Quickstart

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/winstonsmith1897/DantinoX/blob/main/docs/notebooks/01_quickstart.ipynb)

Covers:
- **Level-1** one-liner API (`dx.fit`, `dx.quick_generate`)
- **Level-2** explicit API (`ARParadigm`, `Trainer`, `ModelConfig`)
- Attention variants: MHA · GQA · MLA · Sliding-Window
- FFN variants: SwiGLU · GELU-MLP · Mixture-of-Experts
- Norm types: RMSNorm · LayerNorm
- Positional encodings: RoPE · learned · sinusoidal · none
- Text generation with `Generator` (greedy / top-k / nucleus / streaming)

**Runtime**: GPU (T4 or better recommended)

In [1]:
import os

os.environ['CUDA_VISIBLE_DEVICES'] = '0'

In [2]:
!pip install -q git+https://github.com/winstonsmith1897/DantinoX.git#egg=dantinox[all]

DEPRECATION: git+https://github.com/winstonsmith1897/DantinoX.git#egg=dantinox[all] contains an egg fragment with a non-PEP 508 name pip 25.0 will enforce this behaviour change. A possible replacement is to use the req @ url syntax, and remove the egg fragment. Discussion can be found at https://github.com/pypa/pip/issues/11617


In [3]:
import jax
print('JAX version:', jax.__version__)
print('Devices:', jax.devices())

JAX version: 0.9.1
Devices: [CudaDevice(id=0)]


In [4]:
import urllib.request, os
if not os.path.exists('tiny_shakespeare.txt'):
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt',
        'tiny_shakespeare.txt')
    print('Downloaded tiny_shakespeare.txt')
else:
    print('tiny_shakespeare.txt already present')

tiny_shakespeare.txt already present


## Level 1 — One-liner API

`dx.fit('ar', corpus, **hyperparams)` trains a character-level AR model and returns the run directory.

In [ ]:
import os

os.environ['CUDA_VISIBLE_DEVICES'] = '0'

In [5]:
import dantinox as dx

run_dir = dx.fit(
    'ar', 'tiny_shakespeare.txt',
    dim=256, n_heads=4, head_size=64, num_blocks=4,
    lr=3e-4, epochs=2, batch_size=16, tokenizer_type='bpe'
)
print('Checkpoint saved to:', run_dir)

Epoch 2/2: 100%|██████████| 71/71 [00:01<00:00, 47.27it/s, loss=3.5374]


Checkpoint saved to: runs/20260623_104045


In [6]:
output = dx.quick_generate(run_dir, 'HAMLET:\n', max_new_tokens=200)
print(output)

 HAMLET:ĊPCELERICA:ĊJ hoonarsgis u were! wlIdiny, thenre,ĊThe'tcianted, our; it of psinged thu comeves:Ċ'enell the ward.x-fier you to the wer tood.ĊĊfW will dad thy prohichags?ĊSow, and But oun havend, jecithich,ĊMy; heing! watows it.ĊĊĊSUSINIPIO:ĊLut theAnd you have pa that he, sell is with say, a he wardm don youif, is wid:ĊNhi amPord that her unow turs the c


## Level 2 — Explicit Paradigm API

Separate `ModelConfig` (architecture) and `TrainingConfig` (training) for full control.

In [7]:
model_cfg = dx.ModelConfig(
    dim=256, n_heads=4, head_size=64, num_blocks=4,
    causal=True,
)
train_cfg = dx.TrainingConfig(lr=3e-4, epochs=1, batch_size=16)

paradigm = dx.ARParadigm(model_cfg)
run_dir2 = dx.Trainer(paradigm, train_cfg).fit('tiny_shakespeare.txt')
print('Run dir:', run_dir2)

Epoch 1/1: 100%|██████████| 122/122 [00:14<00:00,  8.28it/s, loss=2.2624]


Run dir: runs/20260623_104149


## Attention Variants

| `attention=` | Description |
|---|---|
| `"mha"` | Multi-Head Attention (default) |
| `"gqa"` | Grouped-Query Attention — add `kv_heads < n_heads` |
| `"mla"` | Multi-Latent Attention (DeepSeek-V2 style) |
| `"mha"` + `sliding_window=True` | Local Sliding-Window Attention |

In [8]:
from flax import nnx
import jax

def param_count(cfg):
    m = dx.ARParadigm(cfg).build_model(nnx.Rngs(0))
    return sum(x.size for x in jax.tree_util.tree_leaves(nnx.state(m, nnx.Param)))

attn_configs = [
    ('MHA',               dx.ModelConfig(dim=256, n_heads=4, head_size=64, num_blocks=4,
                                         vocab_size=200, attention='mha')),
    ('GQA kv_heads=2',   dx.ModelConfig(dim=256, n_heads=4, head_size=64, num_blocks=4,
                                         vocab_size=200, attention='gqa', kv_heads=2)),
    ('GQA kv_heads=1',   dx.ModelConfig(dim=256, n_heads=4, head_size=64, num_blocks=4,
                                         vocab_size=200, attention='gqa', kv_heads=1)),
    ('MLA',               dx.ModelConfig(dim=256, n_heads=4, head_size=64, num_blocks=4,
                                         vocab_size=200, attention='mla')),
    ('SWA window=64',     dx.ModelConfig(dim=256, n_heads=4, head_size=64, num_blocks=4,
                                         vocab_size=200, attention='mha',
                                         sliding_window=True, context_window=64)),
]

for name, cfg in attn_configs:
    n = param_count(cfg)
    print(f'{name:20s}  {n/1e6:.2f}M params')

MHA                   4.26M params
GQA kv_heads=2        4.00M params
GQA kv_heads=1        3.86M params
MLA                   4.86M params
SWA window=64         4.26M params


In [16]:
# Train with GQA — fewer KV heads means faster inference and lower KV-cache memory
gqa_cfg = dx.ModelConfig(
    dim=256, n_heads=4, head_size=64, num_blocks=4,
    attention='gqa', kv_heads=2, ffn='moe', activation='swiglu', moe_latent=True, moe_latent_dim=32
)
gqa_run = dx.Trainer(
    dx.ARParadigm(gqa_cfg),
    dx.TrainingConfig(lr=3e-4, epochs=1, batch_size=16, tokenizer_type='char'),
).fit('tiny_shakespeare.txt', run_dir='/tmp/dx_gqa')
print(dx.quick_generate(gqa_run, 'HAMLET:\n', max_new_tokens=100))

Epoch 1/1: 100%|██████████| 122/122 [00:24<00:00,  5.04it/s, loss=10.6318]


HAMLET:
SF:fHyLfqA M. hevjoyyoadoLhr:


C

Flrf' KAei woEt, co. b t
Zotho wee?y:
Chark sl ith ithortat; d yR


## FFN Variants

| `ffn=` | Description |
|---|---|
| `"mlp"` | SwiGLU MLP (default) |
| `"mlp"` + `ffn_activation="gelu"` | GELU-activated MLP |
| `"moe"` | Mixture-of-Experts — add `n_experts` and `top_k` |

In [10]:
ffn_configs = [
    ('SwiGLU MLP',     dx.ModelConfig(dim=256, n_heads=4, head_size=64, num_blocks=4,
                                      vocab_size=200, ffn='mlp')),
    ('GELU MLP',       dx.ModelConfig(dim=256, n_heads=4, head_size=64, num_blocks=4,
                                      vocab_size=200, ffn='mlp')),
    ('MoE 4 experts',  dx.ModelConfig(dim=256, n_heads=4, head_size=64, num_blocks=4,
                                      vocab_size=200, ffn='moe', n_experts=4, top_k=2)),
    ('MoE 8 experts',  dx.ModelConfig(dim=256, n_heads=4, head_size=64, num_blocks=4,
                                      vocab_size=200, ffn='moe', n_experts=8, top_k=2)),
]

for name, cfg in ffn_configs:
    n = param_count(cfg)
    print(f'{name:20s}  {n/1e6:.2f}M params')

SwiGLU MLP            4.26M params
GELU MLP              4.26M params
MoE 4 experts         13.73M params
MoE 8 experts         26.35M params


## Norm & Positional Encoding Variants

| `norm=` | `pos_encoding=` | Notes |
|---|---|---|
| `"rmsnorm"` | `"rotary"` | Default — RoPE is relative, no pos embedding matrix |
| `"layernorm"` | `"learned"` | Classic BERT-style learned absolute positions |
| `"rmsnorm"` | `"absolute"` | Fixed sinusoidal (Transformer 2017) |
| `"rmsnorm"` | `"none"` | No position info — rely on attention patterns only |

In [ ]:
norm_pos_configs = [
    ('RMSNorm + RoPE',       dx.ModelConfig(dim=256, n_heads=4, head_size=64, num_blocks=4,
                                             vocab_size=200, norm='rmsnorm', pos_encoding='rotary')),
    ('LayerNorm + Learned',  dx.ModelConfig(dim=256, n_heads=4, head_size=64, num_blocks=4,
                                             vocab_size=200, norm='layernorm', pos_encoding='learned')),
    ('RMSNorm + Sinusoidal', dx.ModelConfig(dim=256, n_heads=4, head_size=64, num_blocks=4,
                                             vocab_size=200, norm='rmsnorm', pos_encoding='absolute')),
    ('RMSNorm + None',       dx.ModelConfig(dim=256, n_heads=4, head_size=64, num_blocks=4,
                                             vocab_size=200, norm='rmsnorm', pos_encoding='none')),
]

for name, cfg in norm_pos_configs:
    n = param_count(cfg)
    print(f'{name:30s}  {n/1e6:.2f}M params')

RMSNorm + RoPE                  4.26M params
LayerNorm + Learned             4.39M params
RMSNorm + Sinusoidal            4.26M params
RMSNorm + None                  4.26M params


In [19]:
# Compare RMSNorm vs LayerNorm on the same task
for norm in ('rmsnorm', 'layernorm'):
    cfg = dx.ModelConfig(dim=128, n_heads=2, head_size=64, num_blocks=4,
                         norm=norm)
    rd = dx.Trainer(
        dx.ARParadigm(cfg),
        dx.TrainingConfig(lr=3e-4, epochs=1, batch_size=16),
    ).fit('tiny_shakespeare.txt', run_dir=f'/tmp/dx_{norm}')
    print(f'{norm}: {dx.quick_generate(rd, "HAMLET:", max_new_tokens=60)[:80]}')

Epoch 1/1: 100%|██████████| 122/122 [00:21<00:00,  5.67it/s, loss=2.5385]


rmsnorm: HAMLET: F:f 3deat w.

RbjoyI leoLAi I
T ses.
f it,
PMBo thacou b tQ


Epoch 1/1: 100%|██████████| 122/122 [00:18<00:00,  6.57it/s, loss=2.5627]


layernorm: HAMLET:SF:OS3:NMG M.rhebje w,
MoLHP:


C:
FEL:
HK,
PMBoE::UGo.:L tQ


## Generator — Greedy / Top-k / Nucleus / Streaming

`Generator` wraps any trained model with multiple decoding strategies.

In [20]:
from dantinox.generator import Generator

model = dx.load(run_dir)         # load from run_dir (Level-1 run above)
gen   = Generator(run_dir)       # Generator handles tokenization automatically

for label, kwargs in [
    ('Greedy',          dict(temperature=0.0)),
    ('Top-k (k=40)',    dict(temperature=0.8, top_k=40)),
    ('Nucleus (p=0.9)', dict(temperature=0.9, top_p=0.9)),
]:
    print(f'=== {label} ===')
    print(gen.generate('HAMLET:\n', max_new_tokens=80, **kwargs))
    print()

=== Greedy ===
 HAMLET:Ċ

=== Top-k (k=40) ===
 HAMLET:ĊTen d'sdinghovct and iowath here taul.ĊĊOost CTol to whenied pl, frd,ĊWAREentious, and live wion sincesen mother fairdensili

=== Nucleus (p=0.9) ===
 HAMLET:ĊTen dfatistcow,eedock toerld gowomightonĊA why, what; beart? a wrow ver bemearĊFhe is wrynten, sastereee bls in to,ĊWNow 't



In [21]:
# Streaming — yields tokens one at a time
print('=== Streaming (top-k, k=50) ===')
for chunk in gen.stream('HAMLET:\n', max_new_tokens=80, temperature=0.8, top_k=50):
    print(chunk, end='', flush=True)
print()

=== Streaming (top-k, k=50) ===
ĊHu iss d dulndersptwlelftintay,ĊOar, this ruion:ĊĊGolamay, and bre in of cage wilesse of sheĊAreee but, and heround Giconess


## Analytical FLOPs Profile

`dx.profile` returns FLOPs breakdown without running any training.

In [22]:
for dim, blocks in [(128, 4), (256, 8), (512, 12), (768, 24)]:
    cfg   = dx.ModelConfig(dim=dim, n_heads=max(1, dim//64), head_size=64,
                           num_blocks=blocks, vocab_size=200)
    flops = dx.count_flops(cfg, seq_len=256, batch_size=4)
    n     = param_count(cfg)
    print(f'dim={dim:4d} blocks={blocks:2d}  {n/1e6:6.1f}M params  {flops.total/1e9:.2f} GFLOPs')

dim= 128 blocks= 4     1.1M params  2.47 GFLOPs
dim= 256 blocks= 8     8.5M params  18.36 GFLOPs
dim= 512 blocks=12    50.5M params  106.51 GFLOPs
dim= 768 blocks=24   226.9M params  473.83 GFLOPs
